# 42. Curvature along the stream

This is the last measurement on the list of five that the plasticity literature uses and
this project had not done.

**What curvature means here.** Think of the training loss as a landscape. The model sits at
one point on it. Curvature says how sharply the ground bends around that point. A flat
valley is easy to move around in. A narrow, steep one is not: every step changes the loss a
lot, so training has to take small steps and gets stuck more easily. The claim in the
literature is that a model trained on many tasks in a row drifts into sharp places, and
that this is one reason it stops being able to learn.

**How this notebook measures it.** By a central difference, which needs only forward passes
and no second derivatives. At each point in the stream:

1. Load the model after `t` tasks and attach the same fixed task code the indicators use.
2. Measure the loss on 256 fixed images of a held-out task. Call this `L0`.
3. Pick a random direction. Move the generator a small distance `r` that way, measure the
   loss again (`L_plus`). Move the same distance the other way, measure again (`L_minus`).
4. The curvature in that direction is `2 * ((L_plus + L_minus) / 2 - L0) / r**2`.

Averaging over several random directions gives the mean curvature of the loss surface,
which is the average of the Hessian's eigenvalues. Going both ways and averaging is what
removes the slope: the two first-order terms are equal and opposite, so they cancel, and
what is left is the bend.

**What it does not measure.** The sharpest direction, which is the largest eigenvalue.
That needs Hessian-vector products and a power iteration, which is a much heavier run. The
mean is the cheaper of the two standard numbers and is what this notebook reports. If a
reviewer asks for the largest eigenvalue, that is a separate notebook.

**The reading rule, written down before the run.** The probe does not fall anywhere along
this stream, so if curvature is a trustworthy indicator here it should not rise either. If
it does rise, it joins effective rank, srank and dormant units: a measurement that reports
damage where the direct test finds none. Either answer is a result.

---

## What run 2 changes, and why

Run 1 (`curvature1`, 3 November) ran this notebook with five random directions at each
point and one measurement point. Its raw numbers rose at the end of the stream on all
three seeds, but two things made that unreadable.

**Five directions was not enough.** A random direction is one sample out of a space with
135 million dimensions, and the five draws disagreed badly with each other. The spread
between directions at the end of the stream was 17, 18 and 46 times the spread at the
start. Run 1's noise floor was built from the spread at the start only, which is how every
other measurement in this project builds its floor, and that is correct when the spread
stays steady along the stream. Here it does not. Rebuilding the floor from the spread at
both ends leaves one seed clearing it clearly, one narrowly, and one not at all.

Forty directions cuts the uncertainty by about 2.8 times. On run 1's own spreads that is
enough to turn each of the three seeds into a firm answer: seed 0 and seed 1 would clear
a both-ends floor by 4 and 3.6 times, and seed 2's change would sit well inside it, which
is a definite no rather than an unresolved maybe.

**The measurement point was not a solution.** The loss at the reference point reached
12443. Chance for a ten-class task is 2.303. The reference code belongs to no task and the
pictures come from a task the model never learned, so the model there is not merely bad,
it is somewhere nothing ever put it. Every other indicator in this project reads from that
same point, which is why this notebook used it, and it is the right point for comparing
with them. It is the wrong point for the question the sharpness literature asks, which is
whether the place the model actually settles gets sharper as more tasks arrive.

So run 2 has two arms. The `reference` arm is run 1 repeated with forty directions, and it
stays comparable with the other indicators. The `learned` arm stands the generator at the
code it learned for task 3, the first task in this order, on task 3's own pictures. That
is a solution the stream really reached. Task 3 is learned at boundary 1, so this arm runs
from boundary 2 onward and asks whether that solution grows sharper as the stream goes on.

Both arms load the same model files, so the second arm costs forward passes and no extra
downloads. Both use the same random directions at the same point, so the two arms differ
only by where they stand and what they read.


## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-probe")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.probe import _features_and_logits
from uncle.plasticity_indicators import (REFERENCE_SEED, fixed_image_sets,
                                         indicator_key, ranks, reference_code)

import datetime
import random
import numpy as np
import torch
from dataclasses import replace
from tqdm.auto import tqdm

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)

## 2. Settings

The same study, order and seeds as every other measurement on these models. The probe
points match notebook 40's, so the curvature curve and the probe curve can be read side by
side. Seed 0 kept no middle models, so it has only its two ends.

In [ ]:
STUDY_ID = "indicator_stream_v1"
BUCKET = "uncle-experiments"
ORDER = "order_01"
FINAL_POINT = 8

# A label of its own. Finished rows are matched by identity, so two sets of
# settings sharing one file would mix silently. Run 1 wrote `curvature1`; this
# run has more directions and a second arm, so it writes a new file instead of
# adding rows to that one.
RUN_LABEL = "curvature2"
RESUME_KEY = f"uncle/curvature/{STUDY_ID}/{ORDER}/{RUN_LABEL}.jsonl"
SAVE_EVERY = 20

# -- the two arms -----------------------------------------------------------
#
# `reference` stands the generator at one fixed task code that belongs to no
# task and was never trained, and reads the loss on a held-out task. Every
# other indicator in this project reads from that same point, so this arm is
# the one that can be set beside them. It is run 1 repeated with more
# directions.
#
# `learned` stands at a code the model did learn, on that task's own pictures.
# That is a place the stream actually settled, which is the place the sharpness
# literature means. Run 1 showed why this arm is needed: the loss at the
# reference point reached 12443, where chance for ten classes is 2.303.
REFERENCE_POINTS = {0: [0, 8], 1: [0, 2, 4, 6, 8], 2: [0, 2, 4, 6, 8]}
PROBE_TASK = "2"          # held out, never learned: the reference arm's pictures

# The first task in order 01, so it is finished at boundary 1 and has a trained
# code at every point below. Start and end only: if the ends do not move there
# is nothing in the middle worth paying for.
LEARNED_TASK = "3"
LEARNED_POINTS = {0: [8], 1: [2, 8], 2: [2, 8]}

ARMS = ["reference", "learned"]
POINTS = {"reference": REFERENCE_POINTS, "learned": LEARNED_POINTS}
SEEDS = sorted(set(REFERENCE_POINTS) | set(LEARNED_POINTS))
# Which models have to be loaded: the points either arm asks for, once each.
POINTS_BY_SEED = {seed: sorted({t for arm in ARMS for t in POINTS[arm].get(seed, [])})
                  for seed in SEEDS}

IMAGES_PER_SET = 256

# How far to step when measuring the bend, as a fraction of the generator's own
# size. Three sizes, because a central difference only measures curvature while
# the step is small enough that the surface still looks like a parabola. If the
# three agree the step was small enough. Run 1's three disagreed, but not in one
# direction, which is what too few directions looks like rather than too large a
# step. Forty directions makes this check mean something.
RADII = [0.001, 0.003, 0.01]

# Directions per radius. Run 1 used five, and five was too few: the spread
# between directions at the end of the stream was 17 to 46 times the spread at
# the start, wider than the change being looked for on two of three seeds.
# Forty cuts the uncertainty by about 2.8 times, which on run 1's own spreads is
# enough to make every seed a firm yes or a firm no.
REPEATS = 40

# The base for the direction seeds, unchanged from run 1, so the first five
# draws at each reference point are the same directions run 1 used.
CURVATURE_SEED = 20261104

BATCH_SIZE = 64
SAVE_TO_R2 = True
RELEASE_GPU_WHEN_DONE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"

ROWS_BY_ARM = {arm: sum(len(POINTS[arm].get(seed, [])) for seed in SEEDS)
               * len(RADII) * REPEATS for arm in ARMS}
TOTAL_ROWS = sum(ROWS_BY_ARM.values())
MODELS = sum(1 for seed in SEEDS for t in POINTS_BY_SEED[seed] if t > 0)
print("Study:", STUDY_ID, "| order:", ORDER, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", GPU_NAME)
for arm in ARMS:
    print(f"  {arm:<10} {ROWS_BY_ARM[arm]:>5} measurements")
print(f"{TOTAL_ROWS} curvature measurements in total across {len(SEEDS)} seeds.")
print(f"{MODELS} model files to download, about {MODELS * 0.54:.1f} GB. Both arms "
      "read the same files, so the second arm costs no extra download.")
print("Nothing trains. The cost is the downloads and two forward passes per row.")

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Open the study, build the images, find every model

Every model file is checked before anything runs, from its metadata only, so a missing
boundary stops the notebook in seconds rather than part way through.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
settings = plan["config"]
ORDER_TASKS = list(plan["orders"][ORDER])
CLASSES = settings["classes_per_task"]

if FINAL_POINT != len(ORDER_TASKS):
    raise ValueError(f"FINAL_POINT is {FINAL_POINT} but the order has "
                     f"{len(ORDER_TASKS)} tasks.")
if PROBE_TASK in ORDER_TASKS:
    raise ValueError("The probe task is in the learning order, so it is not held out.")

needed = tuple(ORDER_TASKS + [PROBE_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "tasks")

# The same 256 pictures the indicators read their rank on, and their labels.
# This repeats what fixed_image_sets does, with the labels kept: the same seed
# and the same permutation, so picture for picture it is the same set. Only the
# model differs between two measurements.
pool = tasks[PROBE_TASK]["test"]
stream = torch.Generator().manual_seed(REFERENCE_SEED)
picked = torch.randperm(len(pool), generator=stream)[:IMAGES_PER_SET]
LOSS_IMAGES = torch.stack([pool[int(i)][0] for i in picked])
LOSS_LABELS = torch.tensor([pool[int(i)][1] for i in picked])
image_sets = fixed_image_sets({"probe": pool}, count=IMAGES_PER_SET)
if not torch.equal(LOSS_IMAGES, image_sets["probe"]):
    raise RuntimeError("The labelled batch is not the set the indicators used. "
                       "The curvature and the indicators would describe "
                       "different pictures.")
print(f"Loss measured on {len(LOSS_IMAGES)} held-out images of task {PROBE_TASK}, "
      f"{len(set(LOSS_LABELS.tolist()))} classes present")

# The learned arm's pictures, drawn the same way from the task the model did
# learn. Same seed and same rule as above, so neither arm's batch is cherry
# picked; they differ only in which task they come from.
if LEARNED_TASK not in ORDER_TASKS:
    raise ValueError(f"Task {LEARNED_TASK} is not in order {ORDER}, so no model on "
                     "this stream ever has a trained code for it.")
LEARNED_AT = ORDER_TASKS.index(LEARNED_TASK) + 1
earliest = min(t for points in LEARNED_POINTS.values() for t in points)
if earliest < LEARNED_AT:
    raise ValueError(f"Task {LEARNED_TASK} is finished at boundary {LEARNED_AT}, but "
                     f"LEARNED_POINTS asks for boundary {earliest}, where its code is "
                     "either missing or half trained.")

learned_pool = tasks[LEARNED_TASK]["test"]
stream = torch.Generator().manual_seed(REFERENCE_SEED)
picked = torch.randperm(len(learned_pool), generator=stream)[:IMAGES_PER_SET]
LEARNED_IMAGES = torch.stack([learned_pool[int(i)][0] for i in picked])
LEARNED_LABELS = torch.tensor([learned_pool[int(i)][1] for i in picked])
BATCHES = {"reference": (LOSS_IMAGES, LOSS_LABELS),
           "learned": (LEARNED_IMAGES, LEARNED_LABELS)}
print(f"Learned arm: {len(LEARNED_IMAGES)} held-out images of task {LEARNED_TASK}, "
      f"learned at boundary {LEARNED_AT}")

# Each stream's own record, for the boundary-zero check below.
recorded = {}
for seed in SEEDS:
    key = indicator_key(STUDY_ID, ORDER, seed)
    if store.info(key) is None:
        raise RuntimeError(f"No indicator record for seed {seed} at {key}.")
    text = store.download(key, CACHE / "cache").read_text(encoding="utf-8")
    recorded[seed] = {row["t"]: row for row in
                      (json.loads(line) for line in text.splitlines() if line.strip())}


def job_for(seed):
    return next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{seed}")


def model_key(seed, t):
    """Where that seed's model after t tasks lives in R2."""
    job = job_for(seed)
    if t == FINAL_POINT:
        progress = storage.progress(job)
        if progress is None or progress["status"] != "complete":
            raise RuntimeError(f"Seed {seed}'s stream is not finished in R2.")
        storage.validate_artifact(progress["checkpoint"])
        return progress["checkpoint"]["key"]
    return f"{storage.prefix}/{job['id']}/boundary_{t}.pt"


print("Checking every model this run needs:")
KEYS = {}
for seed in SEEDS:
    for t in POINTS_BY_SEED[seed]:
        if t == 0:
            print(f"  seed {seed}, t = 0: no file needed, the seed rebuilds it")
            continue
        key = model_key(seed, t)
        info = store.info(key)
        if info is None:
            raise RuntimeError(
                f"No model for seed {seed} at boundary {t}. Expected: {key}. "
                "Take that point out of POINTS_BY_SEED.")
        KEYS[(seed, t)] = key
        print(f"  seed {seed}, t = {t}: {info['ContentLength'] / 1e6:>6.1f} MB  {key}")
print("Every model is present.")


def build(seed, t):
    """That seed's model after t tasks.

    The four random seeds are set in the same order `run_job` sets them, so the
    t = 0 model is the one the stream actually started from rather than merely
    one built with the same settings. For t > 0 the saved file overwrites those
    weights anyway, but the seeding is left in so both paths are built the same
    way.

    Every file loaded is checked against where it claims to come from.
    `job_requests_done` counts the learn requests the run had finished when it
    saved, and every request in this study is a learn.
    """
    config = replace(Config(**settings), seed=seed, device=DEVICE,
                     requests=tuple(("learn", task) for task in ORDER_TASKS))
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if config.torch_device.type == "cuda":
        torch.cuda.manual_seed_all(config.seed)
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    if t == 0:
        return target, hypernet, trainer

    path = store.download(KEYS[(seed, t)], CACHE / "cache")
    state = torch.load(path, map_location="cpu", weights_only=False)
    done_here = state.get("job_requests_done")
    if done_here != t:
        raise RuntimeError(
            f"The file for seed {seed} boundary {t} says {done_here} tasks were "
            "finished when it was saved. Measuring it would put a model from "
            "the wrong place in the stream on the curve.")
    if state.get("active_request") is not None:
        raise RuntimeError(
            f"The file for seed {seed} boundary {t} was saved part way through "
            "a task, so it is not a boundary model.")
    checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
    del state
    return target, hypernet, trainer

## 5. Pick up an interrupted run

In [ ]:
PARTIAL = CACHE / "diagnostics" / f"curvature_{RUN_LABEL}.jsonl"
PARTIAL.parent.mkdir(parents=True, exist_ok=True)


def identity(row):
    """Everything that makes one measurement different from another."""
    return (row["arm"], row["seed_label"], row["t"], row["radius"],
            row["repetition"])


results = []
if store.info(RESUME_KEY) is not None:
    saved = store.download(RESUME_KEY, CACHE / "cache")
    for line in saved.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            print("Skipped one unreadable line at the end of the saved file.")
    print(f"Resuming: {len(results)} measurements already finished.")
else:
    print("Starting fresh: nothing saved in R2 yet.")

PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in results), encoding="utf-8")
done = {identity(r) for r in results}
print(f"{TOTAL_ROWS} measurements in the plan, {TOTAL_ROWS - len(done)} still to do.")

## 6. Measure the curvature

One model at a time. For each model the notebook measures the loss where it stands, then
steps a small distance one way and the same distance back the other way and measures again.
The model is returned to its saved weights before every draw, so one step never lands on
top of the last.

At `t = 0` the rebuilt generator is checked against the effective rank the stream recorded
at its own boundary zero. If that does not match, the model is not the one the stream
started from and nothing below would mean anything.

In [ ]:
import gc
import time

from tqdm.auto import tqdm


def direction_seed(seed, t, radius, repetition):
    """A seed of its own for every draw, built from the row's identity.

    The arm is deliberately left out, so both arms see the same directions at
    the same point. They stand in different places on different pictures, so
    nothing is shared between their answers, but it removes one source of
    difference between them. It also means the first five draws of the
    reference arm are the five run 1 used.
    """
    return (CURVATURE_SEED + 100000 * seed + 1000 * t
            + 10 * RADII.index(radius) + repetition)


@torch.no_grad()
def generator_size(hypernet):
    """The length of the generator's weight vector, in double precision.

    A float32 sum over 135 million squared entries loses enough digits to shift
    the step size by more than the effect being measured, so the total is taken
    in double.
    """
    return float(torch.sqrt(sum((p.double() ** 2).sum()
                                for p in hypernet.generator_parameters())))


@torch.no_grad()
def step(hypernet, noise, scale):
    """Add `scale` times a stored direction to every generator parameter."""
    for p, n in zip(hypernet.generator_parameters(), noise):
        p.add_(n * scale)


@torch.no_grad()
def draw_direction(hypernet, seed):
    """One random direction over the whole generator, scaled to unit length.

    Drawn evenly over every entry, so the large parameter tensors take almost
    all of the movement simply because they hold almost all of the entries.
    This is notebook 39's `global` arm, which is the arm that matched the
    stream's own distances. Task codes are not generator parameters, so a code
    is never moved: only the weights that write the target are.
    """
    params = hypernet.generator_parameters()
    gen = torch.Generator(device=params[0].device).manual_seed(seed)
    noise = [torch.randn(p.shape, generator=gen, device=p.device, dtype=p.dtype)
             for p in params]
    length = float(torch.sqrt(sum((n.double() ** 2).sum() for n in noise)))
    for n in noise:
        n.div_(length)
    return noise


def code_for(arm, hypernet):
    """Where this arm stands: a code that belongs to nothing, or a learned one."""
    if arm == "reference":
        return reference_code(hypernet)
    if LEARNED_TASK not in hypernet.task_codes:
        raise RuntimeError(
            f"This model has no code for task {LEARNED_TASK}, so it has not "
            "learned it. The learned arm has nothing to stand on here.")
    return hypernet.task_codes[LEARNED_TASK].detach()


@torch.no_grad()
def loss_at(hypernet, target, trainer, code, images_all, labels_all):
    """Mean cross-entropy on a fixed batch, for the weights right now.

    The network is the one this code generates, so this is a property of the
    generator at that code. Batch-normalisation statistics come from
    `trainer.buffer_template`, the clean set every new task starts from, and the
    model is forced into evaluation mode. Both for the same reason the
    indicators do it: a BatchNorm layer left in training mode rescales using
    whatever batch is in front of it, which would hide what is being measured.
    """
    weights = hypernet.weights_from_code(code)
    modes = hypernet.training, target.training
    hypernet.eval()
    target.eval()
    total = 0.0
    try:
        for start in range(0, len(images_all), BATCH_SIZE):
            images = images_all[start:start + BATCH_SIZE].to(trainer.device)
            labels = labels_all[start:start + BATCH_SIZE].to(trainer.device)
            _, logits = _features_and_logits(hypernet, target, trainer, None, images,
                                             weights=weights,
                                             buffers=trainer.buffer_template)
            total += float(torch.nn.functional.cross_entropy(
                logits, labels, reduction="sum"))
    finally:
        hypernet.train(modes[0])
        target.train(modes[1])
    return total / len(images_all)


def save(row):
    results.append(row)
    done.add(identity(row))
    with PARTIAL.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row) + "\n")


began = time.perf_counter()
new_rows = 0

for seed in SEEDS:
    for t in POINTS_BY_SEED[seed]:
        # Both arms that want this point, so the model is downloaded once.
        work = {arm: [(radius, rep) for radius in RADII for rep in range(REPEATS)
                      if (arm, seed, t, radius, rep) not in done]
                for arm in ARMS if t in POINTS[arm].get(seed, [])}
        work = {arm: todo for arm, todo in work.items() if todo}
        if not work:
            print(f"seed {seed}, t = {t}: already finished, skipping")
            continue

        print(f"\nseed {seed}, t = {t}: loading the model "
              f"({', '.join(work)} arm{'s' if len(work) > 1 else ''})")
        target, hypernet, trainer = build(seed, t)

        # The check the t = 0 rows rest on. The stream recorded its own
        # effective rank at boundary 0; if the rebuild does not reproduce it,
        # this is not the model the stream started from.
        if t == 0:
            here = ranks(hypernet, target, trainer, image_sets,
                         reference_code(hypernet))["effective_rank_probe"]
            there = recorded[seed][0]["effective_rank_probe"]
            gap = abs(here - there) / there
            print(f"  rebuild check: effective rank {here:.4f} against the "
                  f"recorded {there:.4f}, gap {gap * 100:.2f}%")
            if gap > 0.01:
                raise RuntimeError(
                    "The rebuilt generator does not match the one the stream "
                    "started from, so its curvature is not this stream's.")

        size = generator_size(hypernet)
        saved = [p.detach().clone() for p in hypernet.generator_parameters()]

        for arm, todo in work.items():
            code = code_for(arm, hypernet)
            images_all, labels_all = BATCHES[arm]
            centre = loss_at(hypernet, target, trainer, code, images_all, labels_all)
            print(f"  {arm}: generator size {size:.2f}, loss where it stands "
                  f"{centre:.4f} (chance is {float(np.log(CLASSES)):.4f})")

            bar = tqdm(todo, desc=f"{arm} seed {seed} t={t}", unit="measurement")
            for radius, rep in bar:
                bar.set_postfix_str(f"radius {radius:g} rep {rep + 1}")
                distance = radius * size
                noise = draw_direction(hypernet, direction_seed(seed, t, radius, rep))

                step(hypernet, noise, distance)
                plus = loss_at(hypernet, target, trainer, code, images_all, labels_all)
                # Back to the saved weights, then the same distance the other
                # way. Restoring rather than stepping back twice keeps float
                # error from accumulating across the draws at this point.
                with torch.no_grad():
                    for p, s in zip(hypernet.generator_parameters(), saved):
                        p.copy_(s)
                step(hypernet, noise, -distance)
                minus = loss_at(hypernet, target, trainer, code, images_all, labels_all)
                with torch.no_grad():
                    for p, s in zip(hypernet.generator_parameters(), saved):
                        p.copy_(s)
                del noise

                # The central second difference. Going both ways and averaging
                # makes the two slope terms cancel, because they are equal and
                # opposite, so what is left is the bend alone. Dividing by the
                # squared step turns it into the curvature in that direction,
                # which averaged over random directions is the mean eigenvalue
                # of the Hessian.
                rise = 0.5 * (plus + minus) - centre
                curvature = 2.0 * rise / (distance ** 2)
                save({"arm": arm, "seed_label": seed, "t": t, "radius": radius,
                      "repetition": rep, "loss_centre": centre, "loss_plus": plus,
                      "loss_minus": minus, "rise": rise, "step_norm": distance,
                      "generator_norm": size, "curvature": curvature,
                      "run_label": RUN_LABEL, "device": GPU_NAME})
                new_rows += 1
                if new_rows % SAVE_EVERY == 0:
                    store.upload(PARTIAL, RESUME_KEY, overwrite=True)
            bar.close()

        del target, hypernet, trainer, saved
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        store.upload(PARTIAL, RESUME_KEY, overwrite=True)

store.upload(PARTIAL, RESUME_KEY, overwrite=True)
print(f"\n{len(results)} measurements in total, {new_rows} in this session "
      f"({(time.perf_counter() - began) / 60:.1f} minutes).")
missing = [(arm, s, t, r, rep) for arm in ARMS for s in SEEDS
           for t in POINTS[arm].get(s, []) for r in RADII for rep in range(REPEATS)
           if (arm, s, t, r, rep) not in done]
if missing:
    print(f"{len(missing)} still to do. Run the notebook again from the top; "
          "everything already finished is skipped.")
else:
    print("Nothing left to run.")


## 7. Read the result

Three questions, in order.

First, was the step small enough? A central difference measures curvature only while the
surface still looks like a parabola over the step. If the three radii give the same number,
it was. If the number grows with the radius, the measurement is picking up more than
curvature and the smallest radius is the one to trust. Run 1's radii disagreed in no single
direction, which is what too few directions looks like; with forty this check means
something.

Second, does curvature rise along the stream? The floor below is built from the spread at
**both** points, not from the spread at the start alone. Run 1 used the start alone, which
is how every other measurement in this project builds its floor, and here that was wrong:
the spread grows by 17 to 46 times along the stream, so a floor built at the start
understates the uncertainty at the end by a large factor.

Third, is any rise just the loss getting bigger? Cross-entropy curvature carries the scale
of the loss with it. The last column divides one by the other. If curvature rises and
curvature-over-loss does not, the honest reading is that the surface did not get sharper,
the model just sits somewhere with a bigger loss.

The direct probe does not fall anywhere on this stream, so a rise that survives all three
questions puts curvature with effective rank, srank and dormant units.


In [ ]:
import math
import statistics as st


def rows_at(arm, seed, t, radius=None):
    return [r for r in results if r["arm"] == arm and r["seed_label"] == seed
            and r["t"] == t and (radius is None or r["radius"] == radius)]


def summary(arm, seed, t, radius):
    """Mean, spread and count of the curvature readings at one point."""
    rows = rows_at(arm, seed, t, radius)
    if not rows:
        return None
    values = [r["curvature"] for r in rows]
    return {"mean": st.mean(values),
            "sd": st.stdev(values) if len(values) > 1 else float("nan"),
            "n": len(values), "loss": rows[0]["loss_centre"]}


WIDTH = 96
SMALLEST = RADII[0]
print("=" * WIDTH)
print(f"CURVATURE, {STUDY_ID}, order {ORDER}, run {RUN_LABEL}".center(WIDTH))
print(f"{len(results)} measurements | {REPEATS} directions per radius".center(WIDTH))
print("=" * WIDTH)

for arm in ARMS:
    where = (f"reference code, held-out task {PROBE_TASK}" if arm == "reference"
             else f"learned code for task {LEARNED_TASK}, task {LEARNED_TASK}'s images")
    print(f"\n{'#' * WIDTH}")
    print(f"ARM: {arm.upper()}  ({where})")
    print("#" * WIDTH)

    print("\nIS THE STEP SMALL ENOUGH?")
    print("Mean curvature at each radius. These should agree. If they grow with the radius,")
    print("read the smallest radius only and say so.")
    print("-" * WIDTH)
    print(f"{'seed':>5}{'t':>4}" + "".join(f"{'r=' + format(r, 'g'):>16}" for r in RADII))
    print("-" * WIDTH)
    for seed in SEEDS:
        for t in POINTS[arm].get(seed, []):
            cells = []
            for radius in RADII:
                at = summary(arm, seed, t, radius)
                cells.append(f"{at['mean']:>16.4g}" if at else f"{'-':>16}")
            print(f"{seed:>5}{t:>4}" + "".join(cells))
    print("-" * WIDTH)

    print(f"\nCURVATURE ALONG THE STREAM, radius {SMALLEST:g}")
    print("-" * WIDTH)
    print(f"{'seed':>5}{'t':>4}{'loss':>12}{'mean curvature':>18}{'sd':>12}"
          f"{'vs first':>10}{'curv/loss':>12}")
    print("-" * WIDTH)
    for seed in SEEDS:
        points = POINTS[arm].get(seed, [])
        if not points:
            continue
        base = summary(arm, seed, points[0], SMALLEST)
        for t in points:
            at = summary(arm, seed, t, SMALLEST)
            if not at:
                continue
            ratio = at["mean"] / base["mean"] if base and base["mean"] else float("nan")
            print(f"{seed:>5}{t:>4}{at['loss']:>12.4f}{at['mean']:>18.4g}"
                  f"{at['sd']:>12.3g}{ratio:>10.3f}{at['mean'] / at['loss']:>12.3e}")
    print("-" * WIDTH)
    print("`vs first` is curvature at t divided by curvature at this arm's first point.")
    print("`curv/loss` divides out the scale of the loss, which cross-entropy curvature")
    print("carries with it. If `vs first` rises and `curv/loss` does not, the surface did")
    print("not get sharper: the model is just sitting somewhere with a bigger loss.")

    print("\nWHAT THIS SAYS")
    print("-" * WIDTH)
    for seed in SEEDS:
        points = POINTS[arm].get(seed, [])
        if len(points) < 2:
            continue
        first, last = summary(arm, seed, points[0], SMALLEST), summary(arm, seed, points[-1], SMALLEST)
        if not (first and last):
            continue
        # The floor uses the spread at BOTH points. Run 1 used the first point
        # only, which assumes the wobble stays the same along the stream. It
        # does not: it grew by 17 to 46 times, so that floor was far too narrow
        # at the far end and called changes real that were not.
        floor = 1.96 * math.sqrt(first["sd"] ** 2 / first["n"] + last["sd"] ** 2 / last["n"])
        narrow = 1.96 * first["sd"] * (1 / first["n"] + 1 / last["n"]) ** 0.5
        change = last["mean"] - first["mean"]
        scaled = last["mean"] / last["loss"] - first["mean"] / first["loss"]
        if abs(change) <= floor:
            verdict = "NO FINDING, the change is inside the spread of the directions"
        elif change > 0:
            verdict = "sharper at the end, which reads as plasticity loss"
        else:
            verdict = "flatter at the end, the opposite of plasticity loss"
        print(f"  seed {seed}: {first['mean']:.4g} at t = {points[0]}, "
              f"{last['mean']:.4g} at t = {points[-1]}, change {change:+.4g}")
        print(f"           floor from both ends {floor:.4g}"
              f"  (from the first point alone it would read {narrow:.4g})")
        print(f"           {verdict}")
        print(f"           divided by the loss: {scaled:+.3e}, "
              f"{'still up' if scaled > 0 else 'down or flat'}")
    print("-" * WIDTH)

print("\nThe direct probe does not fall anywhere on this stream. A rise that survives the")
print("both-ends floor AND survives dividing by the loss would put curvature with effective")
print("rank, srank and dormant units: reporting damage where the test that trains a task")
print("and counts the score finds none.")


## 8. Save

In [ ]:
import csv, datetime, platform

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT = CACHE / "diagnostics" / f"curvature_{ORDER}_{RUN_LABEL}_{stamp}"
EXPORT.mkdir(parents=True, exist_ok=True)

record = {
    "notebook": "42_curvature",
    "protocol": ("Central difference. The loss is measured on a fixed held-out batch "
                 "under the reference task code, then again a fixed distance either "
                 "side along a random direction. Averaging the two sides cancels the "
                 "slope and leaves the bend. Averaged over random directions this is "
                 "the mean eigenvalue of the Hessian. Two arms: `reference` stands at "
                 "a fixed untrained code on a held-out task, which is where the other "
                 "indicators read; `learned` stands at a code the model trained, on "
                 "that task's own images, which is a solution the stream reached."),
    "limitations": [
        "The mean eigenvalue, not the largest. The largest needs Hessian-vector "
        "products and a power iteration, which is a separate and heavier run.",
        "The reference arm stands at a fixed untrained task code on a task the "
        "model never learned, which is where the other indicators read but is "
        "not a trained minimum. The learned arm fixes that and is the one to "
        "read for the sharpness claim.",
        "The learned arm follows one task's code, the first in the order, and "
        "only at its two ends.",
        "Seed 0 kept no middle models, so it has its two ends only.",
        "Three seeds. The spread between seeds is wide, so this rules out a "
        "large change rather than showing the change is zero.",
    ],
    "arms": ARMS, "points_by_arm": POINTS, "learned_task": LEARNED_TASK,
    "code_revision": HEAD, "study_id": STUDY_ID, "order": ORDER,
    "seeds": SEEDS, "points_by_seed": POINTS_BY_SEED,
    "floor_rule": ("1.96 * sqrt(sd_first**2 / n_first + sd_last**2 / n_last). Run 1 "
                   "used the first point's spread alone, which understated the "
                   "uncertainty at the far end because the spread grows along the "
                   "stream."),
    "probe_task": PROBE_TASK, "images": IMAGES_PER_SET,
    "radii": RADII, "repeats": REPEATS, "curvature_seed": CURVATURE_SEED,
    "environment": {"device": DEVICE, "gpu": GPU_NAME,
                    "torch": torch.__version__, "python": sys.version.split()[0],
                    "platform": platform.platform()},
    "rows": results,
}
(EXPORT / "curvature.json").write_text(json.dumps(record, indent=2) + "\n",
                                       encoding="utf-8")

with (EXPORT / "rows.csv").open("w", newline="", encoding="utf-8") as handle:
    columns = ["arm", "seed_label", "t", "radius", "repetition", "loss_centre",
               "loss_plus", "loss_minus", "rise", "step_norm", "generator_norm",
               "curvature", "device"]
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results)

SAVE_VERIFIED = True
for name in ("curvature.json", "rows.csv"):
    path = EXPORT / name
    try:
        text = path.read_text(encoding="utf-8")
        if name.endswith(".json"):
            json.loads(text)
        elif len(text.splitlines()) != len(results) + 1:
            raise ValueError("row count does not match the measurements")
        print(f"  verified  {name:<18} {path.stat().st_size:>9,} bytes")
    except Exception as problem:
        SAVE_VERIFIED = False
        print(f"  FAILED    {name:<18} {problem}")

SAVED_LOCATION = str(EXPORT)
print("\nSaved locally:", EXPORT)
if SAVE_TO_R2 and SAVE_VERIFIED:
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    for name in ("curvature.json", "rows.csv"):
        store.upload(EXPORT / name, f"{destination}/{name}")
    SAVED_LOCATION = destination
    print("Uploaded to R2:", destination)
elif SAVE_TO_R2:
    print("Not uploading: the local files did not verify.")

## 9. Release the GPU

Only once every file has been written and read back. If anything failed to save the runtime
stays connected, so the run is not lost.

In [ ]:
FINISHED = not [(arm, s, t, r, rep) for arm in ARMS for s in SEEDS
                for t in POINTS[arm].get(s, []) for r in RADII
                for rep in range(REPEATS)
                if (arm, s, t, r, rep) not in done]
if not SAVE_VERIFIED:
    print("NOT releasing the GPU: some output files are missing or unreadable.")
    print("Rerun section 8 before closing this runtime, or the run is lost.")
elif not FINISHED:
    print("Results so far saved to:", SAVED_LOCATION)
    print("NOT releasing the GPU: some measurements are still to do.")
    print("Run the measurement cell again to continue from where it stopped.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Results saved to:", SAVED_LOCATION)
    print("RELEASE_GPU_WHEN_DONE is off, so the runtime stays connected.")
elif not IN_COLAB:
    print("Results saved to:", SAVED_LOCATION)
    print("Not running in Colab, so there is no runtime to release.")
else:
    print("Results saved to:", SAVED_LOCATION)
    print("Everything is finished. Share that path.")
    import gc
    for name in ("tasks", "image_sets"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()